# 🌍 Smart Air Quality Prediction and Pollution Alert System
### Comprehensive Machine Learning Pipeline Grounded in CSE273 Foundations
**Author:** Subrat Panigrahi | **Registration Number:** 26143073 | **Section:** N1  
**Institution:** Lovely School of Computer Science & Engineering, Lovely Professional University  
**Course:** CSE273: Foundations of Machine Learning  
**GitHub Repository:** [Subu2006/Smart-Air-Quality-Prediction-and-Pollution-Alert-System](https://github.com/Subu2006/Smart-Air-Quality-Prediction-and-Pollution-Alert-System)

---
### 📋 Notebook Roadmap:
1. **Environment Setup & Data Ingestion:** Ingest CPCB national air quality dataset (29,531 records across 26 cities).
2. **Exploratory Data Analysis (EDA):** Statistical moments (skewness, kurtosis), criteria pollutant distributions, and Pearson correlation matrices.
3. **Mathematical Foundations:**
   - PCA via Covariance Matrix Eigenvalue Decomposition.
   - Welch's two-sample $t$-test hypothesis testing of seasonal pollution disparities.
   - Gradient Descent optimization comparison (SGD, Momentum, RMSprop, Adam) on AQI regression loss.
4. **Chronological Feature Engineering:** Single-day lag vectors and cyclical harmonic features with zero future leakage.
5. **Candidate Model Training & Chronological Benchmarking:** Baseline, Linear, Ridge, Decision Tree, Random Forest, and Gradient Boosting.
6. **Residual Uncertainty & Error Diagnostics:** 95% empirical forecast confidence intervals and parity plots.
7. **Automated CPCB Alert Engine:** 6-tier CPCB classification and automated health warning triggers.

In [ ]:
# Step 1: Import Core Scientific Libraries
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Insert project src
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from smart_air_quality.data import load_air_quality_csv, POLLUTANTS
from smart_air_quality.features import make_forecasting_frame, split_chronologically
from smart_air_quality.alerts import aqi_category, evaluate_observation
from smart_air_quality.math_foundations import compute_air_quality_pca, perform_seasonal_hypothesis_test, simulate_optimizers_on_aqi_regression
from smart_air_quality.models import train_model, load_model, make_single_prediction

print("✅ All dependencies and modules successfully imported.")

## 1. Data Ingestion & Schema Normalization
We ingest the official Central Pollution Control Board (CPCB) continuous monitoring dataset for India (2015–2020).

In [ ]:
data_path = PROJECT_ROOT / "data/raw/city_day.csv"
if not data_path.exists():
    data_path = PROJECT_ROOT / "data/demo/air_quality_demo.csv"

load_result = load_air_quality_csv(data_path)
df = load_result.frame
audit = load_result.audit

print(f"Total Validated Records: {len(df):,}")
print(f"Cities Monitored: {df['City'].nunique()}")
print(f"Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")
df.head()

## 2. Exploratory Data Analysis & Statistical Moments
We compute univariate moments: mean, median, standard deviation, skewness, and kurtosis to assess distribution asymmetry.

In [ ]:
valid_aqi = df['AQI'].dropna()
mean_v = valid_aqi.mean()
med_v = valid_aqi.median()
skew_v = stats.skew(valid_aqi)
kurt_v = stats.kurtosis(valid_aqi)

print(f"Mean AQI: {mean_v:.2f}")
print(f"Median AQI: {med_v:.2f}")
print(f"Skewness: {skew_v:.2f} (Heavy right-tail)")
print(f"Kurtosis: {kurt_v:.2f} (Leptokurtic, episodic peaks)")

## 3. Mathematical Foundations: PCA, Hypothesis Testing & Optimizers
### 3.1 Principal Component Analysis (Eigenvalue Decomposition)
Decomposing the covariance matrix of 6 criteria pollutants into eigenvalues and eigenvectors.

In [ ]:
pca_results = compute_air_quality_pca(df)
for i, (ev, var) in enumerate(zip(pca_results['eigenvalues'], pca_results['var_explained'])):
    print(f"PC{i+1}: Eigenvalue = {ev:.4f} | Variance Explained = {var*100:.2f}%")

pd.DataFrame(pca_results['loadings'])

### 3.2 Seasonal Hypothesis Testing (Welch's t-test)
Testing $H_0: \mu_{winter} = \mu_{monsoon}$ vs $H_1: \mu_{winter} > \mu_{monsoon}$.

In [ ]:
hypo = perform_seasonal_hypothesis_test(df)
print(f"Winter Mean AQI (95% CI): {hypo['winter_mean']} [{hypo['winter_ci_95'][0]} - {hypo['winter_ci_95'][1]}]")
print(f"Monsoon Mean AQI (95% CI): {hypo['monsoon_mean']} [{hypo['monsoon_ci_95'][0]} - {hypo['monsoon_ci_95'][1]}]")
print(f"Welch t-statistic: {hypo['t_statistic']} | p-value: {hypo['p_value']:.2e}")
print(f"Statistically Significant: {hypo['statistically_significant']}")

### 3.3 Loss Landscape & Optimizer Simulation
Comparing Batch GD, SGD, Momentum, RMSprop, and Adam on the multi-pollutant AQI regression loss function.

In [ ]:
opt_results = simulate_optimizers_on_aqi_regression(df, epochs=40, learning_rate=0.03)
print("Final Convergence Loss by Optimizer:")
for opt_name, final_loss in opt_results['final_loss'].items():
    print(f"  • {opt_name:<10}: {final_loss:.5f}")

## 4. Chronological Feature Engineering & Model Training
Constructing single-day lag inputs and cyclical calendar harmonics, enforcing strict chronological out-of-sample testing.

In [ ]:
model_path = PROJECT_ROOT / "models/air_quality_model.joblib"
meta_path = PROJECT_ROOT / "models/air_quality_model.json"

if not model_path.exists():
    metrics = train_model(data_path, model_path, meta_path)
else:
    metrics = json.loads(meta_path.read_text())

print(f"Selected Best Model: {metrics['model_name']}")
print(f"Residual Error Std: {metrics['residual_std']:.2f} AQI points")
print(f"95% Prediction Interval Margin: ±{metrics['prediction_interval_margin_95']:.2f}")

# Benchmark comparison table
pd.DataFrame(metrics['model_comparison'])[['Model', 'MAE', 'RMSE', 'R2', 'Category_Accuracy', 'Macro_F1']]

## 5. Live Inference & Automated CPCB Alert Evaluation
Generating a 1-day ahead forecast with 95% empirical confidence bounds and automated CPCB health alerting.

In [ ]:
model = load_model(model_path)
sample_input = {
    'AQI': 280.0,
    'PM2.5': 145.0,
    'PM10': 220.0,
    'NO2': 55.0,
    'SO2': 18.0,
    'CO': 2.4,
    'O3': 42.0
}

pred = make_single_prediction(model, 'Delhi', '2026-10-25', sample_input, metrics)
print(f"City: {pred['city']}")
print(f"Observation Date: {pred['observation_date']}")
print(f"Forecast Target Date: {pred['prediction_date']}")
print(f"Predicted Next-Day AQI: {pred['predicted_aqi']:.1f}")
print(f"95% Confidence Interval: [{pred['lower_bound_95']:.1f} - {pred['upper_bound_95']:.1f}]")
print(f"Predicted CPCB Category: {pred['category']}")

alert = evaluate_observation('Delhi', pred['prediction_date'], pred['predicted_aqi'], threshold=200.0)
print(f"Alert Severity: {alert['Severity']}")
print(f"Alert Reason: {alert['Reason']}")